In [26]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error

In [27]:
df = pd.read_csv('Cleaned Data/gurgaon_properties_post_feature_selection.csv')

In [28]:
X = df.drop(columns=['price'])
y = df['price']

In [29]:
# Log-transform target
y_transformed = np.log1p(y)

In [30]:
columns_to_encode = ['sector', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']

In [31]:
# FIX: Add handle_unknown='ignore' to OneHotEncoder
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['property_type', 'bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), columns_to_encode)  # <-- KEY FIX
    ],
    remainder='passthrough'
)


In [32]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', SVR(kernel='rbf'))
])


In [33]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y_transformed, test_size=0.2, random_state=42)


In [34]:
# Fit and predict
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

/home/92c768db-2d10-4625-b98b-4fad701a5f90/.local/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [35]:
# Evaluate (inverse the log transform)
mae = mean_absolute_error(np.expm1(y_test), np.expm1(y_pred))
print(f"MAE: {mae:.4f}")


MAE: 0.6089


In [36]:
# K-Fold Cross Validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
print(f"CV R² Mean: {scores.mean():.4f}")
print(f"CV R² Std:  {scores.std():.4f}")

/home/92c768db-2d10-4625-b98b-4fad701a5f90/.local/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/92c768db-2d10-4625-b98b-4fad701a5f90/.local/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


CV R² Mean: 0.8852
CV R² Std:  0.0136
